# EDA tabular — dataset sintético gaming móvil

Exploración **en tabla** de `gaming_users_raw.csv` (versión estudiantes).

**Objetivo:** conocer estructura, tipos, nulos, cardinalidades y estadísticas descriptivas antes de visualizaciones o clustering.

Referencias: `01-especificacion-dominio-y-esquema.md`, `03-diccionario-datos.md`

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("..").resolve()
CSV_PATH = DATA_DIR / "gaming_users_raw.csv"
MANIFEST_PATH = DATA_DIR / "generation_manifest.json"

CSV_PATH, CSV_PATH.exists()

(WindowsPath('C:/Users/HISC/Documents/Code/UPCHIAPAS/Minería de datos/MAY-AGO2026/Clustering 2/documentation/gaming_users_raw.csv'),
 True)

In [2]:
df = pd.read_csv(CSV_PATH, encoding="utf-8")

for col in ("fecha_registro", "fecha_ultima_sesion"):
    df[col] = pd.to_datetime(df[col], errors="coerce")

BOOL_COLS = [
    "es_pagador", "pertenece_guild",
    "retencion_d1", "retencion_d7", "retencion_d30",
]
CAT_COLS = [
    "plataforma", "pais", "version_app", "modelo_dispositivo_tier",
    "genero_favorito", "tienda_principal",
]
NUM_COLS = [
    c for c in df.columns
    if c not in {"user_id", *BOOL_COLS, *CAT_COLS, "fecha_registro", "fecha_ultima_sesion"}
]

print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")

Filas: 8,080  |  Columnas: 36


## 1. Inventario de columnas

In [3]:
bloque = {
    "user_id": "identificador",
    "fecha_registro": "temporal", "dias_desde_registro": "temporal",
    "fecha_ultima_sesion": "temporal", "dias_inactivo": "temporal",
    "plataforma": "dispositivo", "pais": "dispositivo", "version_app": "dispositivo",
    "modelo_dispositivo_tier": "dispositivo", "ram_gb": "dispositivo",
    "almacenamiento_libre_gb": "dispositivo",
    "sesiones_semanales": "engagement", "minutos_juego_dia": "engagement",
    "streak_dias": "engagement", "hora_pico_sesion": "engagement",
    "partidas_por_sesion": "engagement",
    "gasto_total_mxn": "monetizacion", "gasto_mensual_promedio": "monetizacion",
    "compras_in_app_count": "monetizacion", "ticket_promedio_mxn": "monetizacion",
    "es_pagador": "monetizacion", "tienda_principal": "monetizacion",
    "nivel_jugador": "progresion", "xp_total": "progresion",
    "misiones_completadas": "progresion", "logros_desbloqueados": "progresion",
    "genero_favorito": "progresion",
    "amigos_count": "social", "pertenece_guild": "social",
    "mensajes_chat_semana": "social", "partidas_coop_semana": "social",
    "retencion_d1": "retencion", "retencion_d7": "retencion", "retencion_d30": "retencion",
    "crash_rate_pct": "calidad_tecnica", "latencia_ms_promedio": "calidad_tecnica",
}

inventario = pd.DataFrame({
    "#": range(1, len(df.columns) + 1),
    "columna": df.columns,
    "bloque": [bloque.get(c, "—") for c in df.columns],
    "dtype_pandas": df.dtypes.astype(str).values,
    "no_nulos": df.notna().sum().values,
    "nulos": df.isna().sum().values,
    "pct_nulos": (df.isna().mean() * 100).round(2).values,
    "unicos": df.nunique(dropna=True).values,
})
inventario

,#,columna,bloque,dtype_pandas,no_nulos,nulos,pct_nulos,unicos
0,1,user_id,identificador,object,8080,0,0.00,8080
1,2,fecha_registro,temporal,datetime64[ns],8080,0,0.00,730
2,3,dias_desde_registro,temporal,int64,8080,0,0.00,729
3,4,fecha_ultima_sesion,temporal,datetime64[ns],8077,3,0.04,191
4,5,dias_inactivo,temporal,float64,8076,4,0.05,174
5,6,plataforma,dispositivo,object,8080,0,0.00,6
6,7,pais,dispositivo,object,8080,0,0.00,14
7,8,version_app,dispositivo,float64,8078,2,0.02,5
8,9,modelo_dispositivo_tier,dispositivo,object,8073,7,0.09,3
9,10,ram_gb,dispositivo,float64,7803,277,3.43,115


## 2. Vista rápida de registros

In [4]:
display(df.head(8))
display(df.tail(5))
df.sample(8, random_state=42)

,user_id,fecha_registro,dias_desde_registro,fecha_ultima_sesion,dias_inactivo,plataforma,pais,version_app,modelo_dispositivo_tier,ram_gb,almacenamiento_libre_gb,sesiones_semanales,minutos_juego_dia,streak_dias,hora_pico_sesion,partidas_por_sesion,gasto_total_mxn,gasto_mensual_promedio,compras_in_app_count,ticket_promedio_mxn,es_pagador,tienda_principal,nivel_jugador,xp_total,misiones_completadas,logros_desbloqueados,genero_favorito,amigos_count,pertenece_guild,mensajes_chat_semana,partidas_coop_semana,retencion_d1,retencion_d7,retencion_d30,crash_rate_pct,latencia_ms_promedio
0,USR-000001,2026-05-29,2,2026-05-29,2.00,android,CL,3.20,medio,6.30,40.40,3.74,9.37,12.00,23.00,0.53,0.76,0.76,1.00,0.76,True,google_play,3.00,"4,133.00",15.00,1.00,accion,108.00,True,171.00,42.36,True,True,NaN,1.90,105.10
1,USR-000002,2025-07-14,321,2026-05-26,5.00,ios,MX,3.20,bajo,4.00,6.30,7.33,16.14,20.00,22.00,1.05,2.89,0.27,1.00,2.89,True,app_store,6.00,"68,388.00",150.00,10.00,rpg,26.00,True,7.00,4.04,False,False,NaN,2.25,144.00
2,USR-000003,2024-06-29,701,2026-01-18,133.00,ios,MX,2.80,medio,4.10,35.70,0.26,2.00,0.00,NaN,0.26,0.00,0.00,0.00,0.00,False,NaN,2.00,"6,436.00",12.00,0.00,puzzle,7.00,False,0.00,0.00,False,False,NaN,2.02,82.00
3,USR-000004,2026-03-18,74,2026-05-27,4.00,ios,MX,3.10,alto,10.60,94.90,5.14,14.11,30.00,23.00,0.73,411.95,227.22,19.00,21.68,True,app_store,3.00,"31,103.00",72.00,4.00,estrategia,41.00,True,21.00,11.95,True,False,NaN,0.49,94.80
4,USR-000005,2026-04-25,36,2026-05-20,11.00,ios,BR,3.10,bajo,2.70,NaN,6.00,7.58,1.00,18.00,0.86,0.00,0.00,0.00,0.00,False,NaN,2.00,"1,859.00",15.00,1.00,estrategia,6.00,False,3.00,0.40,True,True,True,4.50,158.50
5,USR-000006,2026-04-20,41,2026-05-31,0.00,android,EC,3.20,medio,6.20,28.30,1.81,3.03,3.00,16.00,0.26,3.92,3.92,0.00,3.92,True,google_play,2.00,"1,553.00",6.00,0.00,rpg,6.00,False,3.00,NaN,False,False,NaN,1.12,114.30
6,USR-000007,2025-03-20,437,2026-05-30,1.00,android,AR,3.10,bajo,3.20,8.70,3.72,8.37,18.00,23.00,0.53,3.42,0.44,4.00,0.86,True,google_play,10.00,"200,139.00",407.00,27.00,rpg,129.00,False,196.00,NaN,True,True,False,3.63,136.10
7,USR-000008,2024-06-19,711,2026-05-23,8.00,android,BR,3.20,medio,6.10,22.30,7.36,16.08,16.00,22.00,1.05,1.04,0.04,1.00,NaN,True,google_play,7.00,"117,660.00",249.00,16.00,puzzle,135.00,True,164.00,22.82,True,False,NaN,1.58,123.30


,user_id,fecha_registro,dias_desde_registro,fecha_ultima_sesion,dias_inactivo,plataforma,pais,version_app,modelo_dispositivo_tier,ram_gb,almacenamiento_libre_gb,sesiones_semanales,minutos_juego_dia,streak_dias,hora_pico_sesion,partidas_por_sesion,gasto_total_mxn,gasto_mensual_promedio,compras_in_app_count,ticket_promedio_mxn,es_pagador,tienda_principal,nivel_jugador,xp_total,misiones_completadas,logros_desbloqueados,genero_favorito,amigos_count,pertenece_guild,mensajes_chat_semana,partidas_coop_semana,retencion_d1,retencion_d7,retencion_d30,crash_rate_pct,latencia_ms_promedio
8075,USR-008076,2025-09-25,248,2026-05-31,0.00,ios,MX,3.10,medio,5.20,31.60,2.27,2.97,1.00,20.00,0.33,0.00,0.00,0.00,NaN,False,NaN,3.00,"13,373.00",30.00,2.00,rpg,11.00,False,2.00,1.58,True,True,False,3.20,90.70
8076,USR-008077,2024-07-05,695,2026-05-31,0.00,android,MX,3.20,medio,7.80,22.60,4.72,6.49,5.00,23.00,0.68,0.41,0.03,0.00,0.41,True,google_play,1.00,"3,757.00",16.00,1.00,casual,4.00,False,0.00,NaN,True,False,NaN,2.91,111.40
8077,USR-008078,2025-01-21,495,2026-05-31,0.00,android,MX,3.20,bajo,3.00,8.20,8.58,26.69,27.00,21.00,1.23,"1,072.81",65.02,8.00,134.10,True,google_play,18.00,"748,856.00","1,514.00",100.00,accion,35.00,False,20.00,6.42,True,True,False,4.57,131.10
8078,USR-008079,2025-01-21,495,2026-02-06,114.00,ios,MX,3.10,bajo,3.20,4.90,0.56,1.46,0.00,NaN,0.89,0.02,0.00,0.00,0.02,True,app_store,1.00,"1,178.00",3.00,0.00,rpg,6.00,False,0.00,NaN,False,False,NaN,4.38,111.60
8079,USR-008080,2024-10-13,595,2026-05-26,5.00,ios,US,3.20,bajo,3.30,5.30,2.53,3.97,1.00,20.00,0.36,0.03,0.00,0.00,0.03,True,app_store,5.00,"58,110.00",121.00,8.00,estrategia,4.00,False,2.00,0.10,False,False,NaN,3.99,111.20


,user_id,fecha_registro,dias_desde_registro,fecha_ultima_sesion,dias_inactivo,plataforma,pais,version_app,modelo_dispositivo_tier,ram_gb,almacenamiento_libre_gb,sesiones_semanales,minutos_juego_dia,streak_dias,hora_pico_sesion,partidas_por_sesion,gasto_total_mxn,gasto_mensual_promedio,compras_in_app_count,ticket_promedio_mxn,es_pagador,tienda_principal,nivel_jugador,xp_total,misiones_completadas,logros_desbloqueados,genero_favorito,amigos_count,pertenece_guild,mensajes_chat_semana,partidas_coop_semana,retencion_d1,retencion_d7,retencion_d30,crash_rate_pct,latencia_ms_promedio
7587,USR-007588,2024-10-07,601,2026-04-08,53.00,ios,US,3.00,bajo,3.60,15.30,0.76,1.52,5.00,23.00,0.11,3.52,0.59,2.00,1.76,True,app_store,3.00,"35,518.00",72.00,4.00,rpg,22.00,False,9.00,2.00,True,False,NaN,4.91,130.20
2069,USR-002070,2025-05-04,392,2026-05-31,0.00,android,AR,3.20,medio,5.50,24.80,11.31,60.04,54.00,22.00,1.62,0.87,0.07,1.00,0.87,True,google_play,14.00,"392,070.00",806.00,53.00,puzzle,9.00,False,7.00,1.13,True,True,False,2.17,129.00
7621,USR-007622,2026-05-05,26,2026-05-27,4.00,android,CR,3.20,bajo,2.00,1.10,5.58,14.41,19.00,22.00,0.80,1.17,1.17,0.00,1.17,True,google_play,2.00,"19,958.00",50.00,3.00,accion,152.00,True,180.00,32.54,True,False,NaN,2.95,145.70
2063,USR-002064,2026-04-01,60,2026-05-21,10.00,android,AR,3.20,bajo,2.70,3.30,3.93,6.39,2.00,22.00,0.56,0.10,0.09,0.00,0.10,True,google_play,2.00,687.00,8.00,0.00,accion,6.00,False,3.00,NaN,False,False,NaN,4.64,172.70
7733,USR-007734,2025-02-12,473,2026-05-24,7.00,android,MX,3.10,medio,5.90,47.00,2.54,3.56,3.00,21.00,0.36,0.00,0.00,0.00,0.00,False,NaN,2.00,"19,243.00",43.00,2.00,puzzle,5.00,False,2.00,0.21,True,False,NaN,2.21,106.30
3437,USR-003438,2026-02-08,112,2026-05-26,5.00,android,UY,3.10,bajo,2.40,10.00,11.05,24.35,14.00,23.00,1.58,0.00,0.00,0.00,0.00,False,NaN,5.00,"25,516.00",73.00,4.00,rpg,22.00,False,10.00,8.21,True,False,NaN,5.11,146.80
6125,USR-006126,2024-11-16,561,2026-05-24,7.00,ios,MX,3.20,bajo,3.60,7.50,6.78,18.56,24.00,21.00,0.97,3.62,0.20,1.00,NaN,True,app_store,7.00,"74,144.00",161.00,10.00,accion,28.00,False,21.00,3.72,True,False,NaN,3.89,123.10
5100,USR-005101,2025-10-08,235,2026-05-25,6.00,ios,MX,3.20,medio,7.00,49.70,18.32,90.26,46.00,21.00,2.62,0.80,0.10,0.00,0.80,True,app_store,9.00,"164,717.00",365.00,24.00,estrategia,14.00,False,2.00,NaN,True,True,True,2.07,85.90


## 3. Resumen de nulos (ordenado)

In [5]:
nulos = (
    df.isna()
    .sum()
    .rename("conteo")
    .to_frame()
    .assign(pct=lambda t: (t["conteo"] / len(df) * 100).round(2))
    .query("conteo > 0")
    .sort_values("conteo", ascending=False)
)
nulos

,conteo,pct
retencion_d30,5064,62.67
tienda_principal,2076,25.69
partidas_coop_semana,1707,21.13
ticket_promedio_mxn,1347,16.67
gasto_mensual_promedio,308,3.81
hora_pico_sesion,278,3.44
ram_gb,277,3.43
almacenamiento_libre_gb,91,1.13
gasto_total_mxn,38,0.47
mensajes_chat_semana,37,0.46


In [6]:
nulos_por_bloque = (
    inventario.groupby("bloque", as_index=False)
    .agg(columnas=("columna", "count"), nulos_totales=("nulos", "sum"), pct_promedio=("pct_nulos", "mean"))
    .assign(pct_promedio=lambda t: t["pct_promedio"].round(2))
    .sort_values("nulos_totales", ascending=False)
)
nulos_por_bloque

,bloque,columnas,nulos_totales,pct_promedio
6,retencion,3,5079,20.95
4,monetizacion,6,3776,7.79
7,social,4,1756,5.44
1,dispositivo,6,377,0.78
2,engagement,5,295,0.73
5,progresion,5,22,0.06
0,calidad_tecnica,2,14,0.08
8,temporal,4,7,0.02
3,identificador,1,0,0.00


## 4. Estadísticas — variables numéricas

In [7]:
desc = df[NUM_COLS].describe().T
desc["cv"] = (desc["std"] / desc["mean"]).replace([np.inf, -np.inf], np.nan).round(2)
desc

,count,mean,std,min,25%,50%,75%,max,cv
dias_desde_registro,"8,080.00",346.22,219.63,1.00,154.00,345.00,540.00,729.00,0.63
dias_inactivo,"8,076.00",14.91,27.88,0.00,3.00,6.00,12.00,198.00,1.87
ram_gb,"7,803.00",5.85,2.80,2.00,3.40,5.70,7.30,31.40,0.48
almacenamiento_libre_gb,"7,989.00",30.04,23.22,1.00,10.00,26.10,42.80,128.00,0.77
sesiones_semanales,"8,076.00",7.73,10.07,0.00,2.70,5.34,9.91,110.33,1.30
minutos_juego_dia,"8,076.00",28.10,97.53,0.00,4.16,11.05,25.29,"2,946.02",3.47
streak_dias,"8,077.00",14.83,15.11,-30.00,3.00,11.00,20.00,70.00,1.02
hora_pico_sesion,"7,802.00",20.47,1.95,13.00,19.00,21.00,22.00,23.00,0.10
partidas_por_sesion,"8,074.00",1.04,0.89,0.02,0.41,0.77,1.40,8.67,0.85
gasto_total_mxn,"8,042.00",57.38,882.55,-986.07,0.00,0.51,1.82,"57,672.89",15.38


In [8]:
percentiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
pct_tabla = df[NUM_COLS].quantile(percentiles).T
pct_tabla.columns = [f"p{int(p*100)}" for p in percentiles]
pct_tabla

,p1,p5,p25,p50,p75,p95,p99
dias_desde_registro,4.00,19.00,154.00,345.00,540.00,694.00,723.00
dias_inactivo,0.00,0.00,3.00,6.00,12.00,89.00,137.25
ram_gb,2.00,2.40,3.40,5.70,7.30,11.00,12.00
almacenamiento_libre_gb,1.00,3.90,10.00,26.10,42.80,75.90,97.11
sesiones_semanales,0.02,0.19,2.70,5.34,9.91,21.05,34.30
minutos_juego_dia,0.18,1.12,4.16,11.05,25.29,106.93,160.91
streak_dias,0.00,0.00,3.00,11.00,20.00,48.00,55.00
hora_pico_sesion,16.00,17.00,19.00,21.00,22.00,23.00,23.00
partidas_por_sesion,0.09,0.16,0.41,0.77,1.40,2.88,4.06
gasto_total_mxn,0.00,0.00,0.00,0.51,1.82,47.19,"1,176.09"


## 5. Frecuencias — variables categóricas y booleanas

In [9]:
def tabla_frecuencia(col: str) -> pd.DataFrame:
    vc = df[col].value_counts(dropna=False)
    out = vc.rename("conteo").to_frame().reset_index()
    out.columns = [col, "conteo"]
    out["pct"] = (out["conteo"] / len(df) * 100).round(2)
    return out


for col in CAT_COLS + BOOL_COLS:
    print(f"\n=== {col} ===")
    display(tabla_frecuencia(col))


=== plataforma ===


,plataforma,conteo,pct
0,android,6014,74.43
1,ios,2047,25.33
2,IOS,6,0.07
3,ios,5,0.06
4,Android,5,0.06
5,android,3,0.04



=== pais ===


,pais,conteo,pct
0,MX,2588,32.03
1,US,952,11.78
2,BR,743,9.20
3,CO,687,8.50
4,AR,635,7.86
5,CL,506,6.26
6,PE,413,5.11
7,ES,391,4.84
8,EC,337,4.17
9,GT,257,3.18



=== version_app ===


...

,retencion_d1,conteo,pct
0,True,5540,68.56
1,False,2533,31.35
2,NaN,7,0.09



=== retencion_d7 ===


,retencion_d7,conteo,pct
0,False,4854,60.07
1,True,3218,39.83
2,NaN,8,0.10



=== retencion_d30 ===


,retencion_d30,conteo,pct
0,NaN,5064,62.67
1,True,1542,19.08
2,False,1474,18.24


## 6. Duplicados e IDs

In [10]:
resumen_ids = pd.DataFrame({
    "metrica": [
        "filas_totales",
        "user_id_unicos",
        "duplicados_exactos_user_id",
        "filas_duplicadas_en_todas_las_columnas",
    ],
    "valor": [
        len(df),
        df["user_id"].nunique(),
        df["user_id"].duplicated().sum(),
        df.duplicated().sum(),
    ],
})
resumen_ids

,metrica,valor
0,filas_totales,8080
1,user_id_unicos,8080
2,duplicados_exactos_user_id,0
3,filas_duplicadas_en_todas_las_columnas,0


In [11]:
cols_similares = [
    "gasto_total_mxn", "sesiones_semanales", "pais", "plataforma", "xp_total"
]
casi_dup = df[df.duplicated(subset=cols_similares, keep=False)].sort_values(cols_similares)
print(f"Registros en grupos casi duplicados: {len(casi_dup)}")
casi_dup.head(20)

Registros en grupos casi duplicados: 104


,user_id,fecha_registro,dias_desde_registro,fecha_ultima_sesion,dias_inactivo,plataforma,pais,version_app,modelo_dispositivo_tier,ram_gb,almacenamiento_libre_gb,sesiones_semanales,minutos_juego_dia,streak_dias,hora_pico_sesion,partidas_por_sesion,gasto_total_mxn,gasto_mensual_promedio,compras_in_app_count,ticket_promedio_mxn,es_pagador,tienda_principal,nivel_jugador,xp_total,misiones_completadas,logros_desbloqueados,genero_favorito,amigos_count,pertenece_guild,mensajes_chat_semana,partidas_coop_semana,retencion_d1,retencion_d7,retencion_d30,crash_rate_pct,latencia_ms_promedio
6717,USR-006718,2024-10-25,583,2026-02-22,98.00,android,US,3.10,medio,5.20,24.80,0.01,0.74,0.00,NaN,0.21,0.00,0.00,0.00,NaN,False,NaN,1.00,"2,154.00",4.00,0.00,rpg,7.00,False,0.00,0.00,False,False,NaN,3.13,79.00
8048,USR-008049,2024-10-25,583,2026-02-22,98.00,android,US,3.10,medio,5.20,24.80,0.01,0.74,0.00,NaN,0.21,0.00,0.00,0.00,NaN,False,NaN,1.00,"2,154.00",4.00,0.00,rpg,7.00,False,0.00,0.00,False,False,NaN,3.13,79.00
764,USR-000765,2024-12-12,535,2026-01-12,139.00,ios,MX,3.20,bajo,2.10,1.40,0.04,2.00,0.00,NaN,0.15,0.00,0.00,0.00,0.00,False,NaN,2.00,"3,530.00",7.00,0.00,casual,3.00,False,0.00,2.23,False,False,NaN,4.73,139.50
8069,USR-008070,2024-12-12,535,2026-01-12,139.00,ios,MX,3.20,bajo,2.10,1.40,0.04,2.00,0.00,NaN,0.15,0.00,0.00,0.00,0.00,False,NaN,2.00,"3,530.00",7.00,0.00,casual,3.00,False,0.00,2.23,False,False,NaN,4.73,139.50
7239,USR-007240,2024-11-22,555,2026-01-12,139.00,android,BR,3.00,bajo,2.00,1.00,0.14,1.02,0.00,23.00,0.08,0.00,0.00,0.00,NaN,False,NaN,3.00,"9,460.00",18.00,1.00,casual,3.00,False,0.00,0.00,False,False,NaN,4.47,157.20
8015,USR-008016,2024-11-22,555,2026-01-12,139.00,android,BR,3.00,bajo,2.00,1.00,0.14,1.02,0.00,23.00,0.08,0.00,0.00,0.00,NaN,False,NaN,3.00,"9,460.00",18.00,1.00,casual,3.00,False,0.00,0.00,False,False,NaN,4.47,157.20
3521,USR-003522,2026-03-07,85,2026-04-08,53.00,android,MX,3.10,medio,6.50,36.00,0.14,2.00,0.00,21.00,0.32,0.00,0.00,0.00,0.00,False,NaN,2.00,700.00,1.00,0.00,puzzle,3.00,False,1.00,0.02,True,True,True,2.06,108.70
8035,USR-008036,2026-03-07,85,2026-04-08,53.00,android,MX,3.10,medio,6.50,36.00,0.14,2.00,0.00,21.00,0.32,0.00,0.00,0.00,0.00,False,NaN,2.00,700.00,1.00,0.00,puzzle,3.00,False,1.00,0.02,True,True,True,2.06,108.70
4202,USR-004203,2025-12-10,172,2026-05-13,18.00,ios,MX,3.10,bajo,2.20,4.60,0.70,1.01,1.00,19.00,0.10,0.00,0.00,0.00,0.00,False,NaN,1.00,"9,316.00",19.00,1.00,accion,2.00,False,1.00,3.34,True,True,False,3.86,130.80
8072,USR-008073,2025-12-10,172,2026-05-13,18.00,ios,MX,3.10,bajo,2.20,4.60,0.70,1.01,1.00,19.00,0.10,0.00,0.00,0.00,0.00,False,NaN,1.00,"9,316.00",19.00,1.00,accion,2.00,False,1.00,3.34,True,True,False,3.86,130.80


## 7. Reglas de calidad (conteo de violaciones)

In [12]:
reglas = {
    "gasto_negativo": df["gasto_total_mxn"] < 0,
    "streak_negativo": df["streak_dias"] < 0,
    "minutos_mayor_1440": df["minutos_juego_dia"] > 1440,
    "ram_mayor_16": df["ram_gb"] > 16,
    "fecha_ultima_antes_registro": df["fecha_ultima_sesion"] < df["fecha_registro"],
    "pagador_sin_gasto": (df["es_pagador"] == True) & (df["gasto_total_mxn"].fillna(0) <= 0),
    "gasto_sin_pagador": (df["es_pagador"] == False) & (df["gasto_total_mxn"].fillna(0) > 0),
    "compras_sin_gasto": (df["compras_in_app_count"].fillna(0) > 0) & (df["gasto_total_mxn"].fillna(0) == 0),
    "ios_con_google_play": (df["plataforma"] == "ios") & (df["tienda_principal"] == "google_play"),
}

calidad = (
    pd.Series({k: int(v.sum()) for k, v in reglas.items()}, name="conteo")
    .to_frame()
    .assign(pct=lambda t: (t["conteo"] / len(df) * 100).round(3))
    .sort_values("conteo", ascending=False)
)
calidad

,conteo,pct
pagador_sin_gasto,54,0.67
compras_sin_gasto,45,0.56
gasto_sin_pagador,37,0.46
fecha_ultima_antes_registro,29,0.36
gasto_negativo,21,0.26
ram_mayor_16,17,0.21
minutos_mayor_1440,13,0.16
streak_negativo,13,0.16
ios_con_google_play,0,0.00


## 8. Tablas por segmento

In [13]:
segmento_plataforma = (
    df.groupby("plataforma", observed=True)
    .agg(
        usuarios=("user_id", "count"),
        sesiones_mediana=("sesiones_semanales", "median"),
        gasto_mediana=("gasto_total_mxn", "median"),
        xp_mediana=("xp_total", "median"),
        pct_pagador=("es_pagador", lambda s: (s == True).mean() * 100),
    )
    .round(2)
)
segmento_plataforma

,usuarios,sesiones_mediana,gasto_mediana,xp_mediana,pct_pagador
plataforma,,,,,
Android,5,7.13,0.00,"208,426.00",80.00
IOS,6,4.16,1.62,"138,574.50",83.33
android,6014,5.38,0.52,"35,508.50",74.26
android,3,1.08,0.21,"40,256.00",100.00
ios,2047,5.21,0.50,"34,197.50",72.74
ios,5,5.21,0.55,"35,912.00",60.00


In [14]:
segmento_genero = (
    df.groupby("genero_favorito", observed=True)
    .agg(
        usuarios=("user_id", "count"),
        sesiones_media=("sesiones_semanales", "mean"),
        gasto_media=("gasto_total_mxn", "mean"),
        nivel_mediana=("nivel_jugador", "median"),
    )
    .round(2)
    .sort_values("usuarios", ascending=False)
)
segmento_genero

,usuarios,sesiones_media,gasto_media,nivel_mediana
genero_favorito,,,,
rpg,2108,8.60,64.89,5.00
puzzle,1662,6.34,17.57,3.00
accion,1620,8.64,45.71,5.00
casual,1415,5.80,80.18,3.00
estrategia,1275,9.07,86.55,6.00


In [15]:
pivot_pais_plataforma = pd.crosstab(df["pais"], df["plataforma"], margins=True)
pivot_pais_plataforma

plataforma,Android,IOS,android,android,ios,ios,All
pais,,,,,,,
AR,2,2,478,0,152,1,635
BO,0,0,57,0,6,0,63
BR,0,0,614,0,129,0,743
CL,0,0,352,1,152,1,506
CO,0,0,551,0,136,0,687
CR,0,0,176,0,75,1,252
EC,0,0,286,0,51,0,337
ES,0,0,238,0,153,0,391
GT,0,0,228,0,29,0,257


In [16]:
metricas_segmento = ["sesiones_semanales", "gasto_total_mxn", "xp_total", "dias_inactivo"]
pivot_tier = df.groupby("modelo_dispositivo_tier", observed=True)[metricas_segmento].agg(["count", "mean", "median", "std"]).round(2)
pivot_tier

sesiones_semanales                   gasto_total_mxn  \
                                     count mean median   std           count   
modelo_dispositivo_tier                                                        
alto                                  1572 7.69   5.45  8.99            1564   
bajo                                  2818 7.64   5.36  9.81            2807   
medio                                 3679 7.81   5.29 10.70            3664   

                                            xp_total                         \
                         mean median    std    count         mean    median   
modelo_dispositivo_tier                                                       
alto                    44.35   0.53 292.26     1570 1,145,358.28 37,497.00   
bajo                    55.77   0.51 941.73     2818   137,814.64 33,702.00   
medio                   64.27   0.52 996.93     3682   379,130.24 35,532.50   

                                      dias_inactivo                     
                                  std         count  mean median   std  
modelo_dispositivo_tier                                                 
alto                    22,208,133.77          1571 14.80   6.00 27.58  
bajo                       253,729.77          2817 15.09   6.00 28.84  
medio                    7,705,697.17          3681 14.82   6.00 27.29

## 9. Monetización — percentiles y coherencia

In [17]:
pagadores = df[df["es_pagador"] == True]
no_pagadores = df[df["es_pagador"] == False]

monet_resumen = pd.DataFrame({
    "segmento": ["todos", "pagadores", "no_pagadores"],
    "n": [len(df), len(pagadores), len(no_pagadores)],
    "pct": [
        100.0,
        len(pagadores) / len(df) * 100,
        len(no_pagadores) / len(df) * 100,
    ],
    "gasto_mediana": [
        df["gasto_total_mxn"].median(),
        pagadores["gasto_total_mxn"].median(),
        no_pagadores["gasto_total_mxn"].median(),
    ],
    "gasto_p95": [
        df["gasto_total_mxn"].quantile(0.95),
        pagadores["gasto_total_mxn"].quantile(0.95),
        no_pagadores["gasto_total_mxn"].quantile(0.95),
    ],
    "compras_mediana": [
        df["compras_in_app_count"].median(),
        pagadores["compras_in_app_count"].median(),
        no_pagadores["compras_in_app_count"].median(),
    ],
}).round(2)
monet_resumen

,segmento,n,pct,gasto_mediana,gasto_p95,compras_mediana
0,todos,8080,100.00,0.51,47.19,0.00
1,pagadores,5970,73.89,1.00,332.84,0.00
2,no_pagadores,2110,26.11,0.00,0.00,0.00


## 10. Engagement, social y retención

In [18]:
engagement_pagador = (
    df.groupby("es_pagador", observed=True)[
        ["sesiones_semanales", "minutos_juego_dia", "streak_dias", "partidas_por_sesion"]
    ]
    .agg(["count", "mean", "median"])
    .round(2)
)
engagement_pagador

sesiones_semanales             minutos_juego_dia               \
                        count mean median             count  mean median   
es_pagador                                                                 
False                    2110 7.42   3.50              2108 23.22   5.06   
True                     5966 7.84   6.02              5968 29.82  14.20   

           streak_dias              partidas_por_sesion              
                 count  mean median               count mean median  
es_pagador                                                           
False             2108 10.42   3.00                2108 0.86   0.51  
True              5969 16.39  14.00                5966 1.11   0.87

In [19]:
social_guild = (
    df.groupby("pertenece_guild", observed=True)[
        ["amigos_count", "mensajes_chat_semana", "partidas_coop_semana"]
    ]
    .agg(["count", "mean", "median"])
    .round(2)
)
social_guild

amigos_count              mensajes_chat_semana               \
                       count  mean median                count  mean median   
pertenece_guild                                                               
False                   6680 18.30  11.00                 6676 12.91   3.00   
True                    1388 69.40  45.00                 1364 91.32  28.50   

                partidas_coop_semana               
                               count  mean median  
pertenece_guild                                    
False                           4986  4.39   2.08  
True                            1385 18.89  16.12

In [20]:
retencion_tabla = pd.DataFrame({
    "metrica": ["retencion_d1", "retencion_d7", "retencion_d30"],
    "no_nulos": [df[c].notna().sum() for c in ["retencion_d1", "retencion_d7", "retencion_d30"]],
    "pct_true": [
        (df[c] == True).mean() * 100 for c in ["retencion_d1", "retencion_d7", "retencion_d30"]
    ],
}).round(2)
retencion_tabla

,metrica,no_nulos,pct_true
0,retencion_d1,8073,68.56
1,retencion_d7,8072,39.83
2,retencion_d30,3016,19.08


In [21]:
pd.crosstab(df["retencion_d1"], df["retencion_d7"], dropna=False, margins=True)

retencion_d7,False,True,NaN,All
retencion_d1,,,,
False,2529,0,4,"2,533.00"
True,2322,3214,4,"5,540.00"
NaN,3,4,0,NaN
All,4854,3218,0,"8,080.00"


## 11. Correlación entre variables numéricas (tabla)

In [22]:
corr = df[NUM_COLS].corr(numeric_only=True).round(3)
corr

,dias_desde_registro,dias_inactivo,ram_gb,almacenamiento_libre_gb,sesiones_semanales,minutos_juego_dia,streak_dias,hora_pico_sesion,partidas_por_sesion,gasto_total_mxn,gasto_mensual_promedio,compras_in_app_count,ticket_promedio_mxn,nivel_jugador,xp_total,misiones_completadas,logros_desbloqueados,amigos_count,mensajes_chat_semana,partidas_coop_semana,crash_rate_pct,latencia_ms_promedio
dias_desde_registro,1.00,0.08,-0.01,-0.00,0.03,0.01,0.07,-0.01,0.06,0.01,-0.08,0.04,0.03,0.33,0.01,0.29,0.29,0.06,0.04,0.05,0.04,0.01
dias_inactivo,0.08,1.00,-0.01,-0.01,-0.23,-0.10,-0.38,-0.01,-0.31,-0.02,-0.05,-0.16,-0.08,-0.31,-0.02,-0.22,-0.22,-0.23,-0.16,-0.22,0.00,0.01
ram_gb,-0.01,-0.01,1.00,0.74,0.01,0.01,0.01,-0.00,0.01,-0.01,0.00,-0.01,0.01,0.01,0.02,0.01,0.01,0.01,0.01,0.01,-0.72,-0.56
almacenamiento_libre_gb,-0.00,-0.01,0.74,1.00,0.02,0.02,0.02,-0.00,0.03,-0.00,0.00,0.00,0.02,0.01,0.02,0.01,0.01,-0.00,-0.01,-0.00,-0.70,-0.56
sesiones_semanales,0.03,-0.23,0.01,0.02,1.00,0.26,0.47,0.00,0.62,-0.00,0.02,0.08,0.05,0.38,0.03,0.37,0.37,0.06,0.02,0.05,-0.00,-0.01
minutos_juego_dia,0.01,-0.10,0.01,0.02,0.26,1.00,0.29,-0.01,0.34,0.01,-0.00,-0.00,0.00,0.24,0.03,0.23,0.23,0.01,-0.01,0.00,-0.01,0.00
streak_dias,0.07,-0.38,0.01,0.02,0.47,0.29,1.00,-0.03,0.74,0.04,0.06,0.17,0.12,0.80,0.06,0.72,0.72,0.10,0.01,0.07,-0.01,-0.01
hora_pico_sesion,-0.01,-0.01,-0.00,-0.00,0.00,-0.01,-0.03,1.00,-0.01,-0.00,-0.01,0.01,0.00,-0.02,-0.01,-0.03,-0.03,0.01,0.01,0.01,0.01,0.07
partidas_por_sesion,0.06,-0.31,0.01,0.03,0.62,0.34,0.74,-0.01,1.00,0.02,0.03,0.09,0.07,0.64,0.07,0.58,0.59,0.06,-0.01,0.04,-0.01,-0.01
gasto_total_mxn,0.01,-0.02,-0.01,-0.00,-0.00,0.01,0.04,-0.00,0.02,1.00,0.29,0.19,0.30,0.04,0.01,0.02,0.02,0.01,-0.00,0.03,0.00,-0.01


In [23]:
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .rename("r")
    .reset_index()
    .rename(columns={"level_0": "var_a", "level_1": "var_b"})
    .assign(abs_r=lambda t: t["r"].abs())
    .sort_values("abs_r", ascending=False)
    .head(20)
    .drop(columns="abs_r")
)
pairs

TypeError: Series.reset_index() got an unexpected keyword argument 'names'

## 12. Muestras estratificadas (inspección manual)

In [ ]:
cols_vista = [
    "user_id", "plataforma", "pais", "sesiones_semanales", "gasto_total_mxn",
    "xp_total", "dias_inactivo", "es_pagador", "pertenece_guild", "genero_favorito",
]

print("=== Top gasto (posibles whales) ===")
display(df.nlargest(8, "gasto_total_mxn")[cols_vista])

print("=== Mayor inactividad (posible churn) ===")
display(df.nlargest(8, "dias_inactivo")[cols_vista])

print("=== Mayor engagement ===")
display(df.nlargest(8, "sesiones_semanales")[cols_vista])

print("=== Mayor actividad social ===")
display(df.nlargest(8, "mensajes_chat_semana")[cols_vista])

## 13. Manifiesto de generación (metadatos)

In [ ]:
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, encoding="utf-8") as f:
        manifest = json.load(f)
    pd.DataFrame([manifest]).T.rename(columns={0: "valor"})
else:
    print("No se encontró generation_manifest.json")

## 14. Resumen ejecutivo (tabla)

In [ ]:
resumen_final = pd.DataFrame({
    "hallazgo": [
        "Registros y columnas",
        "Columnas con al menos un nulo",
        "Duplicados exactos (todas las columnas)",
        "Violaciones de calidad detectadas (suma)",
        "Correlación sesiones–minutos",
        "% usuarios pagadores",
        "% con guild",
    ],
    "valor": [
        f"{len(df):,} × {df.shape[1]}",
        int((df.isna().sum() > 0).sum()),
        int(df.duplicated().sum()),
        int(calidad["conteo"].sum()),
        round(df[["sesiones_semanales", "minutos_juego_dia"]].corr().iloc[0, 1], 3),
        round((df["es_pagador"] == True).mean() * 100, 2),
        round((df["pertenece_guild"] == True).mean() * 100, 2),
    ],
})
resumen_final